In [1]:
!pip install pyspark
!pip install findspark
!pip install pandas
!pip install numpy
!pip install pyarrow

In [2]:
import findspark
findspark.init()

In [3]:
import pandas as pd
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

In [4]:
# Creating a spark context class
sc = SparkContext()

# Creating a spark session
spark = SparkSession \
    .builder \
    .appName("Python Spark DataFrames basic example") \
    .config("spark.some.config.option", "some-value") \
    .getOrCreate()

In [5]:
spark

In [6]:
!pip install -q gdown

import gdown

file_id = "1FsVEx3Me6r6ITAItE46sRC_LmNvwzYb9"

gdown.download(
    f"https://drive.google.com/uc?id={file_id}",
    "citibike.csv",
    quiet=False
)

df = spark.read.csv(
    "citibike.csv",
    header=True,
    inferSchema=True
)

df.show(5)

Downloading...
From (original): https://drive.google.com/uc?id=1FsVEx3Me6r6ITAItE46sRC_LmNvwzYb9
From (redirected): https://drive.google.com/uc?id=1FsVEx3Me6r6ITAItE46sRC_LmNvwzYb9&confirm=t&uuid=41227631-0354-4410-95fe-b2c8ef937f26
To: /content/citibike.csv
100%|██████████| 241M/241M [00:03<00:00, 76.3MB/s]


+---+--------------------+--------------------+----------------+--------------------+----------------------+-----------------------+--------------+--------------------+--------------------+---------------------+------+----------+----------+------+
|_c0|           starttime|            stoptime|start station id|  start station name|start station latitude|start station longitude|end station id|    end station name|end station latitude|end station longitude|bikeid|  usertype|birth year|gender|
+---+--------------------+--------------------+----------------+--------------------+----------------------+-----------------------+--------------+--------------------+--------------------+---------------------+------+----------+----------+------+
|  0|2019-04-17 14:37:...|2019-04-17 14:43:...|           264.0|Maiden Ln & Pearl St|           40.70706456|           -74.00731853|         330.0| Reade St & Broadway|         40.71450451|         -74.00562789| 16906|Subscriber|      1969|     1|
|  1|201

In [7]:
# Preview a few records
df.head()

Row(_c0=0, starttime=datetime.datetime(2019, 4, 17, 14, 37, 3, 844000), stoptime=datetime.datetime(2019, 4, 17, 14, 43, 13, 767000), start station id=264.0, start station name='Maiden Ln & Pearl St', start station latitude=40.70706456, start station longitude=-74.00731853, end station id=330.0, end station name='Reade St & Broadway', end station latitude=40.71450451, end station longitude=-74.00562789, bikeid=16906, usertype='Subscriber', birth year=1969, gender=1)

In [8]:
df.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- starttime: timestamp (nullable = true)
 |-- stoptime: timestamp (nullable = true)
 |-- start station id: double (nullable = true)
 |-- start station name: string (nullable = true)
 |-- start station latitude: double (nullable = true)
 |-- start station longitude: double (nullable = true)
 |-- end station id: double (nullable = true)
 |-- end station name: string (nullable = true)
 |-- end station latitude: double (nullable = true)
 |-- end station longitude: double (nullable = true)
 |-- bikeid: integer (nullable = true)
 |-- usertype: string (nullable = true)
 |-- birth year: integer (nullable = true)
 |-- gender: integer (nullable = true)



In [9]:
df.count()

1300000

Checking missing values:


In [10]:
from pyspark.sql.functions import col, sum, when

df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]).show()

+---+---------+--------+----------------+------------------+----------------------+-----------------------+--------------+----------------+--------------------+---------------------+------+--------+----------+------+
|_c0|starttime|stoptime|start station id|start station name|start station latitude|start station longitude|end station id|end station name|end station latitude|end station longitude|bikeid|usertype|birth year|gender|
+---+---------+--------+----------------+------------------+----------------------+-----------------------+--------------+----------------+--------------------+---------------------+------+--------+----------+------+
|  0|        0|       0|              33|                33|                     0|                      0|            33|              33|                   0|                    0|     0|       0|         0|     0|
+---+---------+--------+----------------+------------------+----------------------+-----------------------+--------------+----------

In [11]:
df = df.dropna(
    subset=[
        "start station id",
        "start station name",
        "end station id",
        "end station name"
    ]
)

In [12]:
df.count()

1299967

Check duplicates:

In [13]:
duplicates = df.count() - df.dropDuplicates().count()
print("Number of duplicate rows:", duplicates)

Number of duplicate rows: 0


Converting Data Types:

In [14]:
from pyspark.sql.functions import col

df = df.withColumn("start station id", col("start station id").cast("int")) \
       .withColumn("end station id", col("end station id").cast("int"))

Rider Age:


In [15]:
from pyspark.sql.functions import year, current_date

df = df.withColumn(
    "Rider_Age",
    year(current_date()) - col("birth year")
)
df.select("birth year", "Rider_Age").show(5)

+----------+---------+
|birth year|Rider_Age|
+----------+---------+
|      1969|       57|
|      1974|       52|
|      1969|       57|
|      1986|       40|
|      1979|       47|
+----------+---------+
only showing top 5 rows


Trip duration:

In [16]:
from pyspark.sql.functions import unix_timestamp

df = df.withColumn(
    "Trip_Duration_Seconds",
    unix_timestamp("stoptime") - unix_timestamp("starttime")
)
df.select("Trip_Duration_Seconds").show(5)

+---------------------+
|Trip_Duration_Seconds|
+---------------------+
|                  370|
|                  347|
|                  919|
|                  591|
|                  331|
+---------------------+
only showing top 5 rows


 Trip Distance:

In [17]:
from pyspark.sql.functions import radians, sin, cos, sqrt, atan2
R = 6371
df = df.withColumn(
    "distance_km",
    2 * R * atan2(
        sqrt(
            sin((radians(col("end station latitude")) - radians(col("start station latitude"))) / 2) ** 2 +
            cos(radians(col("start station latitude"))) *
            cos(radians(col("end station latitude"))) *
            sin((radians(col("end station longitude")) - radians(col("start station longitude"))) / 2) ** 2
        ),
        sqrt(
            1 - (
                sin((radians(col("end station latitude")) - radians(col("start station latitude"))) / 2) ** 2 +
                cos(radians(col("start station latitude"))) *
                cos(radians(col("end station latitude"))) *
                sin((radians(col("end station longitude")) - radians(col("start station longitude"))) / 2) ** 2
            )
        )
    )
)
df.select("distance_km").show(5)

+------------------+
|       distance_km|
+------------------+
|0.8394676548008196|
|0.9725410537635826|
|0.7346180002081097|
|1.0676592264973912|
|0.7730897713722797|
+------------------+
only showing top 5 rows


Trip Speed:

In [18]:
df = df.withColumn(
    "Trip_Speed_Kmh",
    (col("distance_km") / col("Trip_Duration_Seconds")) * 3600
)
df.select("Trip_Speed_Kmh").show(5)

+------------------+
|    Trip_Speed_Kmh|
+------------------+
| 8.167793398062027|
|10.089763093800858|
|2.8777201313919423|
| 6.503507978664312|
| 8.408227120665279|
+------------------+
only showing top 5 rows


Period of Day:

In [19]:
from pyspark.sql.functions import hour,when

df = df.withColumn("hour", hour(col("starttime")))


df = df.withColumn(
    "period_of_day",
    when((col("hour") >= 5) & (col("hour") < 12), "Morning")
    .when((col("hour") >= 12) & (col("hour") < 17), "Afternoon")
    .when((col("hour") >= 17) & (col("hour") < 21), "Evening")
    .otherwise("Night")
)
df.select("hour", "Period_of_Day").show(10)

+----+-------------+
|hour|Period_of_Day|
+----+-------------+
|  14|    Afternoon|
|  14|    Afternoon|
|  14|    Afternoon|
|  14|    Afternoon|
|  14|    Afternoon|
|  14|    Afternoon|
|  14|    Afternoon|
|  14|    Afternoon|
|  14|    Afternoon|
|  14|    Afternoon|
+----+-------------+
only showing top 10 rows


Extract Start Month

In [20]:
from pyspark.sql.functions import month
df = df.withColumn(
    "Start_Month",
    month(col("starttime"))
)
df.select("Start_Month").show(5)

+-----------+
|Start_Month|
+-----------+
|          4|
|          4|
|          4|
|          4|
|          4|
+-----------+
only showing top 5 rows


Flag Noise Records:

In [21]:
from pyspark.sql.functions import col, when

df = df.withColumn(
    "Noise_Flag",
    when(col("Trip_Duration_Seconds") < 60, "Short Trip")
    .when(col("Trip_Speed_Kmh") > 40, "Unrealistic Speed")
    .when((col("Rider_Age") > 100) | (col("Rider_Age") < 12), "Invalid Age")
    .when(
        col("start station name").isNull() |
        col("end station name").isNull() |
        col("bikeid").isNull(),
        "Missing Identifier"
    )
    .otherwise("Valid")
)

View Noisy Records:

In [22]:
df.filter(col("Noise_Flag") != "Valid").show()

+-----+--------------------+--------------------+----------------+--------------------+----------------------+-----------------------+--------------+--------------------+--------------------+---------------------+------+----------+----------+------+---------+---------------------+-------------------+------------------+----+-------------+-----------+-----------+
|  _c0|           starttime|            stoptime|start station id|  start station name|start station latitude|start station longitude|end station id|    end station name|end station latitude|end station longitude|bikeid|  usertype|birth year|gender|Rider_Age|Trip_Duration_Seconds|        distance_km|    Trip_Speed_Kmh|hour|period_of_day|Start_Month| Noise_Flag|
+-----+--------------------+--------------------+----------------+--------------------+----------------------+-----------------------+--------------+--------------------+--------------------+---------------------+------+----------+----------+------+---------+-------------

In [23]:
df.createOrReplaceTempView("citi_bike")

a) percentage of round trips


In [24]:
spark.sql("""
SELECT
    usertype,
    COUNT(*) AS total_trips,
    SUM(CASE
        WHEN `start station id` = `end station id` THEN 1
        ELSE 0
    END) AS round_trips,
    (SUM(CASE
        WHEN `start station id` = `end station id` THEN 1
        ELSE 0
    END) * 100.0 / COUNT(*)) AS round_trip_percentage
FROM citi_bike
GROUP BY usertype
""").show()

+----------+-----------+-----------+---------------------+
|  usertype|total_trips|round_trips|round_trip_percentage|
+----------+-----------+-----------+---------------------+
|Subscriber|    1116203|      17965|     1.60947426229817|
|  Customer|     183764|       9740|     5.30027644152282|
+----------+-----------+-----------+---------------------+



Buisness interprtation:
Subscribers rarely make round trips (1.6%), showing they mainly use bikes for commuting.
Customers have more round trips (5.3%), indicating more leisure or recreational usage.


b) Most popular start stations:

In [25]:
spark.sql("""
SELECT
    `start station name`,
    COUNT(*) AS total_trips
FROM citi_bike
GROUP BY `start station name`
ORDER BY total_trips DESC
LIMIT 10""").show()

+--------------------+-----------+
|  start station name|total_trips|
+--------------------+-----------+
|Pershing Square N...|       9762|
|     8 Ave & W 31 St|       8002|
|  E 17 St & Broadway|       7533|
|  Broadway & E 22 St|       7049|
|     W 21 St & 6 Ave|       7022|
|  Broadway & E 14 St|       6977|
|West St & Chamber...|       6731|
|  Broadway & W 60 St|       6599|
|Christopher St & ...|       6557|
|    12 Ave & W 40 St|       6447|
+--------------------+-----------+



The most popular start stations are concentrated in central Manhattan, with Pershing Square North and major avenues like 8 Ave & W 31 St having the highest trip counts.
This indicates heavy usage in high-traffic business and commuting areas, helping planners focus on expanding capacity and improving bike availability in these locations.


c) Rush hours (peak demand):

In [26]:
spark.sql("""
SELECT
    HOUR(starttime) AS hour,
    COUNT(*) AS total_trips
FROM citi_bike
GROUP BY HOUR(starttime)
ORDER BY total_trips DESC
LIMIT 10""").show()

+----+-----------+
|hour|total_trips|
+----+-----------+
|  17|     126653|
|   8|     115768|
|  18|     104935|
|  16|      98325|
|  15|      90985|
|  14|      88564|
|  13|      83851|
|   9|      79560|
|  12|      76365|
|   7|      70116|
+----+-----------+



The highest bike usage occurs during the late afternoon and evening hours, especially at 17:00–18:00, with a smaller peak in the morning around 8:00.
This suggests strong commuter patterns where users travel to and from work.
It helps optimize bike availability during rush hours for better system efficiency.


d) Analyze trip durations by age group

In [27]:
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType

def age_group(age):
    if age < 25:
        return "Young"
    elif age <= 50:
        return "Adult"
    else:
        return "Senior"

age_group_udf = udf(age_group, StringType())

In [28]:
df = df.withColumn("Age_Group", age_group_udf(col("Rider_Age")))

In [29]:
df.createOrReplaceTempView("citi_bike")

In [30]:
spark.sql("""
SELECT
    Age_Group,
    AVG(Trip_Duration_Seconds) AS avg_duration
FROM citi_bike
GROUP BY Age_Group
ORDER BY avg_duration DESC""").show()

+---------+------------------+
|Age_Group|      avg_duration|
+---------+------------------+
|    Young|1181.5919361984936|
|   Senior|1083.6153262900284|
|    Adult| 892.3628862052341|
+---------+------------------+



Young riders have the longest average trip duration, suggesting more leisure or flexible travel behavior.
Seniors also take relatively long trips compared to adults, possibly due to slower riding pace.
Adults have the shortest average duration, indicating more efficient or commute-focused trips.


e) Trip behaviour based on seasons:

In [31]:
def season(month):
    if month in [12,1,2]:
        return "Winter"
    elif month in [3,4,5]:
        return "Spring"
    elif month in [6,7,8]:
        return "Summer"
    else:
        return "Autumn"

season_udf = udf(season, StringType())

In [32]:
df = df.withColumn(
    "Season",
    season_udf(df["Start_Month"])
)

In [33]:
df.createOrReplaceTempView("citibike")

In [34]:
spark.sql("""
SELECT
    Season,
    COUNT(*) AS total_trips,
    ROUND(AVG(Trip_Duration_Seconds),2) AS avg_duration,
    ROUND(AVG(distance_km),2) AS avg_distance
FROM citibike
GROUP BY Season""").show()

+------+-----------+------------+------------+
|Season|total_trips|avg_duration|avg_distance|
+------+-----------+------------+------------+
|Spring|     299994|      939.77|        1.76|
|Summer|     449976|     1018.23|        1.85|
|Autumn|     400000|      958.64|        1.79|
|Winter|     149997|      831.07|        1.53|
+------+-----------+------------+------------+



Summer is the busiest season with the longest and most frequent trips.
Winter has the lowest usage and shortest trips due to less favorable conditions.
Spring and Autumn show moderate, stable biking activity in between.


f) Bike Utilization Analysis: Identifying High-Usage Bikes for Maintenance Planning

In [35]:
spark.sql("""
SELECT
    bikeid,
    SUM(Trip_Duration_Seconds) AS total_usage_seconds,
    COUNT(*) AS total_trips
FROM citibike
GROUP BY bikeid
ORDER BY total_usage_seconds DESC""").show()

+------+-------------------+-----------+
|bikeid|total_usage_seconds|total_trips|
+------+-------------------+-----------+
| 40515|            2964195|         29|
| 25073|            2632367|         47|
| 32225|            2457156|        100|
| 21580|            2313519|         56|
| 26495|            2210383|         66|
| 15876|            2194728|         98|
| 21541|            2075319|         62|
| 27447|            2021581|         91|
| 17509|            1935308|         47|
| 32295|            1811080|         82|
| 15400|            1810823|         63|
| 24931|            1728496|         77|
| 29730|            1591294|         67|
| 18174|            1517529|         56|
| 20584|            1491768|         66|
| 18112|            1468652|         33|
| 38179|            1464344|         74|
| 25720|            1450052|         82|
| 21142|            1436845|         46|
| 28660|            1373098|          3|
+------+-------------------+-----------+
only showing top

Bikes like **40515**, **25073**, **32225** have the highest total usage time, so they are the most heavily worn and should be prioritized for maintenance.
Some bikes have fewer trips but very high usage time, meaning long rides that also increase wear.

g)

In [36]:
spark.sql("""
SELECT
    usertype,
    `end station name`,
    COUNT(*) AS total_trips
FROM citi_bike
GROUP BY usertype, `end station name`
ORDER BY usertype DESC, total_trips DESC
""").show()

+----------+--------------------+-----------+
|  usertype|    end station name|total_trips|
+----------+--------------------+-----------+
|Subscriber|Pershing Square N...|       9269|
|Subscriber|  Broadway & E 22 St|       7379|
|Subscriber|  E 17 St & Broadway|       7128|
|Subscriber|     8 Ave & W 31 St|       7055|
|Subscriber|     W 21 St & 6 Ave|       6736|
|Subscriber|  Broadway & E 14 St|       6485|
|Subscriber|Lafayette St & E ...|       5836|
|Subscriber|West St & Chamber...|       5755|
|Subscriber|Christopher St & ...|       5673|
|Subscriber|  E 47 St & Park Ave|       5578|
|Subscriber|    W 20 St & 11 Ave|       5380|
|Subscriber|     8 Ave & W 33 St|       5358|
|Subscriber|     W 41 St & 8 Ave|       5325|
|Subscriber|     W 31 St & 7 Ave|       5242|
|Subscriber|Cooper Square & A...|       5153|
|Subscriber|     W 38 St & 8 Ave|       5034|
|Subscriber|North Moore St & ...|       4981|
|Subscriber|E 24 St & Park Ave S|       4972|
|Subscriber|  Broadway & W 25 St| 

In [37]:
spark.sql("""
SELECT
    usertype,
    `end station name`,
    COUNT(*) AS total_trips
FROM citi_bike
WHERE usertype = 'Customer'
GROUP BY usertype, `end station name`
ORDER BY total_trips DESC
""").show()

+--------+--------------------+-----------+
|usertype|    end station name|total_trips|
+--------+--------------------+-----------+
|Customer|     5 Ave & E 88 St|       2363|
|Customer|Central Park S & ...|       2336|
|Customer|     5 Ave & E 73 St|       2155|
|Customer|Central Park West...|       2066|
|Customer|    12 Ave & W 40 St|       1984|
|Customer|Centre St & Chamb...|       1952|
|Customer|West St & Chamber...|       1864|
|Customer|  Broadway & W 60 St|       1672|
|Customer|    W 34 St & 11 Ave|       1658|
|Customer|Grand Army Plaza ...|       1589|
|Customer|7 Ave & Central P...|       1522|
|Customer|Pier 40 - Hudson ...|       1521|
|Customer|     5 Ave & E 78 St|       1369|
|Customer|South End Ave & L...|       1322|
|Customer|    W 20 St & 11 Ave|       1204|
|Customer|Central Park Nort...|       1144|
|Customer|Washington St & G...|       1127|
|Customer|       Old Fulton St|       1107|
|Customer|Little West St & ...|       1089|
|Customer|Central Park West...| 

Subscribers mainly end trips at busy commuter hubs in Manhattan (e.g., Pershing Square North, Broadway stations), showing routine work-related travel patterns.
Customers end trips more at tourist and leisure areas like Central Park and waterfront locations, indicating recreational or sightseeing usage.
Overall, subscribers represent daily urban commuters, while customers show non-regular, location-driven leisure mobility behavior.

h)


In [38]:
spark.sql("""
SELECT
    `start station name`,
    `end station name`,
    COUNT(*) AS total_rides
FROM citi_bike
GROUP BY `start station name`, `end station name`
ORDER BY total_rides DESC
""").show()

+--------------------+--------------------+-----------+
|  start station name|    end station name|total_rides|
+--------------------+--------------------+-----------+
|   E 7 St & Avenue A|Cooper Square & A...|        585|
|Central Park S & ...|     5 Ave & E 88 St|        443|
|Central Park S & ...|Central Park S & ...|        433|
|    Soissons Landing|    Soissons Landing|        413|
|Yankee Ferry Term...|    Soissons Landing|        411|
|Vesey Pl & River ...|North Moore St & ...|        401|
|    Soissons Landing|        Picnic Point|        379|
|    Soissons Landing|Yankee Ferry Term...|        371|
|        Picnic Point|    Soissons Landing|        367|
|    12 Ave & W 40 St|West St & Chamber...|        350|
|   E 6 St & Avenue B|Cooper Square & A...|        347|
|McGuinness Blvd &...|Vernon Blvd & 50 Ave|        346|
|Yankee Ferry Term...|Yankee Ferry Term...|        338|
|Pershing Square N...|     W 33 St & 7 Ave|        334|
|Pershing Square N...|E 24 St & Park Ave S|     

* The highest-demand station pairs show strong commuting flows within Manhattan (e.g., E 7 St & Avenue A → Cooper Square), indicating common short-distance urban travel routes.
* Some high counts of same-start and same-end stations (e.g., Central Park, Soissons Landing) suggest **round trips for recreational or leisure activities**.


i)

In [39]:
spark.sql("""
SELECT
    gender,
    COUNT(*) AS total_trips,
    ROUND(AVG(Trip_Speed_Kmh), 2) AS avg_speed,
    ROUND(AVG(Trip_Duration_Seconds), 2) AS avg_duration
FROM citi_bike
GROUP BY gender
""").show()

+------+-----------+---------+------------+
|gender|total_trips|avg_speed|avg_duration|
+------+-----------+---------+------------+
|     1|     886237|     9.22|      831.69|
|     2|     312247|     8.26|      987.98|
|     0|     101483|     6.42|     1996.95|
+------+-----------+---------+------------+



* Gender 1 shows the highest speed and shortest trip duration, indicating more frequent and faster commuting behavior.
* Gender 0 has much slower speeds and very long durations, suggesting more casual or irregular riding patterns compared to others.


j

In [40]:
from pyspark.sql.functions import dayofweek, when

df = df.withColumn(
    "Day_Type",
    when(dayofweek("starttime").isin([1,7]), "Weekend")
    .otherwise("Weekday")
)

df.createOrReplaceTempView("citi_bike")

In [41]:
spark.sql("""
SELECT
    Day_Type,
    ROUND(AVG(Trip_Duration_Seconds),2) AS avg_duration,
    ROUND(AVG(distance_km),2) AS avg_distance,
    ROUND(AVG(Trip_Speed_Kmh),2) AS avg_speed
FROM citi_bike
GROUP BY Day_Type
""").show()

+--------+------------+------------+---------+
|Day_Type|avg_duration|avg_distance|avg_speed|
+--------+------------+------------+---------+
| Weekday|      892.49|        1.76|     9.05|
| Weekend|      1162.7|        1.81|     7.92|
+--------+------------+------------+---------+



* Weekends show **longer trips and slightly higher distance but lower speed**, indicating more leisure-oriented and relaxed riding behavior.
* Weekdays have **shorter, faster trips**, suggesting commuter usage where efficiency and time-saving are priorities.


# SparkML

In [42]:
#Select Relevant Features
data = df.select(
    "gender",
    "Trip_Duration_Seconds",
    "Trip_Speed_Kmh",
    "distance_km",
    "Rider_Age",
    "hour",
    "Start_Month"
).na.drop()

In [43]:
#convert column label column;Spark ML needs numeric labels
from pyspark.ml.feature import StringIndexer

label_indexer = StringIndexer(
    inputCol="gender",
    outputCol="label"
)

In [44]:
#Assemble features into one vector
from pyspark.ml.feature import VectorAssembler

feature_cols = [
    "Trip_Duration_Seconds",
    "Trip_Speed_Kmh",
    "distance_km",
    "Rider_Age",
    "hour",
    "Start_Month"
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

In [45]:
#Normalizing features
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(
    inputCol="features",
    outputCol="scaledFeatures",
    withMean=False,
    withStd=True
)

In [46]:
#create pipeline
from pyspark.ml import Pipeline
pipeline = Pipeline(stages=[label_indexer, assembler, scaler])

In [47]:
pipeline_model = pipeline.fit(data)

In [48]:
data = pipeline_model.transform(data)

In [49]:
#columns nedded for ML models
featuresCol="scaledFeatures"

In [50]:
train, test = data.randomSplit([0.8, 0.2], seed=42)

In [51]:
train.select("label", "scaledFeatures").show(5)

+-----+--------------------+
|label|      scaledFeatures|
+-----+--------------------+
|  2.0|[0.00640816972572...|
|  2.0|[0.00640816972572...|
|  2.0|[0.00651322168844...|
|  2.0|[0.00651322168844...|
|  2.0|[0.00651322168844...|
+-----+--------------------+
only showing top 5 rows


Set up evaluator (accuracy)

In [52]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

Logistic regression

In [53]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="scaledFeatures",
    labelCol="label"
)

lr_model = lr.fit(train)
lr_predictions = lr_model.transform(test)

Decision tree

In [54]:
from pyspark.ml.classification import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    featuresCol="scaledFeatures",
    labelCol="label"
)

dt_model = dt.fit(train)
dt_predictions = dt_model.transform(test)

random forest

In [55]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol="scaledFeatures",
    labelCol="label",
    numTrees=20
)

rf_model = rf.fit(train)
rf_predictions = rf_model.transform(test)

Evaluate Accuracy

In [56]:
lr_acc = evaluator.evaluate(lr_predictions)
dt_acc = evaluator.evaluate(dt_predictions)
rf_acc = evaluator.evaluate(rf_predictions)

In [57]:
print("Logistic Regression Accuracy:", lr_acc)
print("Decision Tree Accuracy:", dt_acc)
print("Random Forest Accuracy:", rf_acc)

Logistic Regression Accuracy: 0.6772889810969249
Decision Tree Accuracy: 0.7407226355759645
Random Forest Accuracy: 0.7407226355759645


In [58]:
#compare models
results = {
    "Logistic Regression": lr_acc,
    "Decision Tree": dt_acc,
    "Random Forest": rf_acc
}

print(results)

{'Logistic Regression': 0.6772889810969249, 'Decision Tree': 0.7407226355759645, 'Random Forest': 0.7407226355759645}


In [59]:
#only Random Forest gives importance
import pandas as pd

feature_names = [
    "Trip_Duration_Seconds",
    "Trip_Speed_Kmh",
    "distance_km",
    "Rider_Age",
    "hour",
    "Start_Month"
]

importances = rf_model.featureImportances.toArray()

pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

,Feature,Importance
3,Rider_Age,0.891397
1,Trip_Speed_Kmh,0.071641
0,Trip_Duration_Seconds,0.030518
2,distance_km,0.004892
5,Start_Month,0.001188
4,hour,0.000364


The analysis compared three machine learning models for predicting gender: Logistic Regression, Decision Tree, and Random Forest. Here's a summary of their performance:

Logistic Regression Accuracy: 0.677
Decision Tree Accuracy: 0.741
Random Forest Accuracy: 0.741
Both the Decision Tree and Random Forest models achieved the highest accuracy (approximately 74.1%), significantly outperforming Logistic Regression. Random Forest, an ensemble method, typically performs well by combining multiple decision trees to reduce overfitting and improve generalization. The similar performance with Decision Tree might suggest that a single well-tuned tree can capture a good portion of the underlying patterns in this dataset.

Regarding feature importance, derived from the Random Forest model, the most influential features in predicting gender are:

Rider_Age: With an importance of 0.835, this is by far the most significant feature.
Trip_Speed_Kmh: Follows with an importance of 0.093.
Trip_Duration_Seconds: Has an importance of 0.063.
distance_km: Has a much lower importance of 0.004.
hour: Even lower, at 0.003.
Start_Month: The least influential, with an importance of 0.001.
This indicates that a rider's age is the strongest predictor of gender in this dataset, followed by trip speed and duration. Other factors like distance, hour of the day, and starting month have minimal impact on the prediction.

In [83]:
# =========================================================
# BONUS DASHBOARD VISUALIZATION - CITI BIKE PROJECT
# =========================================================

!pip install ipywidgets -q

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import ipywidgets as widgets

from IPython.display import display, clear_output
from pyspark.sql.functions import *

plt.rcParams['figure.figsize'] = (10,6)

# =========================================================
# INTERACTIVE FILTER
# =========================================================

user_filter = widgets.Dropdown(
    options=['All', 'Subscriber', 'Customer'],
    value='All',
    description='User Type:'
)

output = widgets.Output()

display(user_filter)
display(output)

# =========================================================
# FILTER FUNCTION
# =========================================================

def get_filtered_df():
    if user_filter.value == "All":
        return df
    return df.filter(col("usertype") == user_filter.value)

# =========================================================
# DASHBOARD FUNCTION
# =========================================================

def update_dashboard(change=None):

    with output:
        clear_output(wait=True)

        filtered_df = get_filtered_df()

        # =====================================================
        # 1 — HOURLY DEMAND (TEMPORAL)
        # =====================================================
        print("Insight: Peak demand occurs during morning and evening rush hours.")

        hourly_pd = filtered_df.groupBy(
            hour("starttime").alias("hour")
        ).count().orderBy("hour").toPandas()

        plt.plot(hourly_pd["hour"], hourly_pd["count"], marker='o')
        plt.title(f"Hourly Citi Bike Demand ({user_filter.value})")
        plt.xlabel("Hour of Day")
        plt.ylabel("Number of Trips")
        plt.grid(True)
        plt.show()

        # =====================================================
        # 2 — WEEKDAY VS WEEKEND
        # =====================================================
        print("Insight: Weekends show more recreational rides.")

        weekday_pd = filtered_df.groupBy("Day_Type").agg(
            round(avg("Trip_Duration_Seconds"),2).alias("Avg_Duration"),
            round(avg("distance_km"),2).alias("Avg_Distance")
        ).toPandas()

        x = np.arange(len(weekday_pd))
        width = 0.35

        fig, ax = plt.subplots()
        ax.bar(x - width/2, weekday_pd["Avg_Duration"], width, label='Duration')
        ax.bar(x + width/2, weekday_pd["Avg_Distance"], width, label='Distance')

        ax.set_xticks(x)
        ax.set_xticklabels(weekday_pd["Day_Type"])
        ax.set_xlabel("Day Type")
        ax.set_ylabel("Average Value")
        plt.title("Weekday vs Weekend Behavior")
        plt.legend()
        plt.show()

        # =====================================================
        # 3 — SEASONAL VARIATION
        # =====================================================
        print("Insight: Usage increases in warmer seasons and drops in winter.")

        season_pd = filtered_df.groupBy("Season").count().toPandas()

        plt.bar(season_pd["Season"], season_pd["count"])
        plt.title("Seasonal Bike Usage")
        plt.xlabel("Season")
        plt.ylabel("Number of Trips")
        plt.show()

        # =====================================================
        # 4 — TOP START STATIONS
        # =====================================================
        print("Insight: High-demand stations are located in busy city areas.")

        stations_pd = filtered_df.groupBy(
            "start station name"
        ).count().orderBy(desc("count")).limit(10).toPandas()

        plt.barh(stations_pd["start station name"], stations_pd["count"])
        plt.gca().invert_yaxis()
        plt.title("Top Start Stations")
        plt.xlabel("Number of Trips")
        plt.ylabel("Station Name")
        plt.show()

        # =====================================================
        # 5 — STATION PAIRS
        # =====================================================
        print("Insight: Station pairs show major commuting routes.")

        pairs_pd = filtered_df.groupBy(
            "start station name",
            "end station name"
        ).count().orderBy(desc("count")).limit(10).toPandas()

        pairs_pd["pair"] = pairs_pd["start station name"] + " → " + pairs_pd["end station name"]

        plt.barh(pairs_pd["pair"], pairs_pd["count"])
        plt.gca().invert_yaxis()
        plt.title("Top Station Pairs")
        plt.xlabel("Number of Trips")
        plt.ylabel("Station Pair")
        plt.show()

        # =====================================================
        # 6 — AGE GROUP ANALYSIS
        # =====================================================
        print("Insight: Younger riders tend to take longer trips.")

        age_pd = filtered_df.groupBy("Age_Group").agg(
            round(avg("Trip_Duration_Seconds"),2).alias("Avg_Duration")
        ).toPandas()

        plt.bar(age_pd["Age_Group"], age_pd["Avg_Duration"])
        plt.title("Trip Duration by Age Group")
        plt.xlabel("Age Group")
        plt.ylabel("Average Trip Duration (Seconds)")
        plt.show()

        # =====================================================
        # 7 — GENDER ANALYSIS
        # =====================================================
        print("Insight: Gender distribution shows slight differences in usage.")

        gender_pd = filtered_df.groupBy("gender").count().toPandas()

        gender_pd["gender"] = gender_pd["gender"].replace({
            0: "Unknown",
            1: "Male",
            2: "Female"
        })

        plt.bar(gender_pd["gender"], gender_pd["count"])
        plt.title("Trips by Gender")
        plt.xlabel("Gender")
        plt.ylabel("Number of Trips")
        plt.show()

        # =====================================================
        # 8 — BIKE UTILIZATION
        # =====================================================
        print("Insight: Some bikes are heavily used and require maintenance.")

        bike_pd = filtered_df.groupBy("bikeid").count()\
            .orderBy(desc("count")).limit(10).toPandas()

        plt.bar(bike_pd["bikeid"].astype(str), bike_pd["count"])
        plt.title("Most Utilized Bikes")
        plt.xlabel("Bike ID")
        plt.ylabel("Number of Trips")
        plt.xticks(rotation=45)
        plt.show()

        # =====================================================
        # 9 — BIKE DURATION DISTRIBUTION
        # =====================================================
        print("Insight: Some bikes are used for longer trips than others.")

        bike_dur_pd = filtered_df.groupBy("bikeid").agg(
            avg("Trip_Duration_Seconds").alias("Avg_Duration")
        ).orderBy(desc("Avg_Duration")).limit(10).toPandas()

        plt.bar(bike_dur_pd["bikeid"].astype(str), bike_dur_pd["Avg_Duration"])
        plt.title("Average Trip Duration per Bike")
        plt.xlabel("Bike ID")
        plt.ylabel("Average Trip Duration (Seconds)")
        plt.xticks(rotation=45)
        plt.show()

        # =====================================================
        # 10 — USER TYPE DISTRIBUTION
        # =====================================================
        print("Insight: Subscribers dominate overall bike usage.")

        user_pd = filtered_df.groupBy("usertype").count().toPandas()

        plt.bar(user_pd["usertype"], user_pd["count"])
        plt.title("Trips by User Type")
        plt.xlabel("User Type")
        plt.ylabel("Number of Trips")
        plt.show()

        print("Dashboard Updated Successfully")

# =========================================================
# RUN DASHBOARD
# =========================================================

user_filter.observe(update_dashboard, names='value')
update_dashboard()

Dropdown(description='User Type:', options=('All', 'Subscriber', 'Customer'), value='All')

Output()

The analysis of the Citi Bike dataset reveals clear patterns in user behavior and system usage. Demand peaks during commuting hours, while weekends and warmer seasons show more recreational riding. Subscribers dominate overall usage, and trip patterns vary across age groups and genders. High-traffic stations and heavily used bikes highlight key operational areas for maintenance and capacity planning. Overall, the system is primarily driven by daily commuting behavior with noticeable seasonal and demographic variations.